# ANPR (Indian number plates): train the character CNN in Google Colab

**How to use:** first upload this project to your GitHub account. Then
1. set `REPO_URL` below to your repository,
2. choose **Runtime → Change runtime type → T4 GPU** (optional, makes training about 10x faster),
3. choose **Runtime → Run all**.

The notebook trains the baseline (reference Kaggle CNN) and the improved CNN, compares them, exports the
improved model to ONNX, and tests the full pipeline on the sample images.

In [ ]:
REPO_URL = "https://github.com/jaypee83/anpr-indian-license-plate"   # <- change to your repository
!rm -rf anpr-indian-license-plate && git clone -q $REPO_URL anpr-indian-license-plate
%cd anpr-indian-license-plate
!ls

## 1 · Install libraries and fonts
The TeX Gyre fonts are the *held-out* test fonts (never used in training).

In [ ]:
!apt-get -qq install -y fonts-texgyre fonts-liberation fonts-dejavu > /dev/null
!pip install -q -r training/requirements-train.txt

## 2 · Train both models and evaluate (about 1-2 min on GPU, 13 min on CPU)

In [ ]:
!python training/train_cnn.py --epochs 15

## 3 · Results

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
m = json.load(open("models/metrics.json"))
keys = ["original_val_acc", "heldout_font_char_acc", "e2e_segmentation_ok", "e2e_plate_exact_raw",
        "e2e_plate_exact_with_format_decoding"]
df = pd.DataFrame({name: {k: f"{r[k]:.1%}" for k in keys} | {"real plate read as": r["real_sample_plate"]["read"]}
                   for name, r in m["models"].items()})
display(df)
print("Held-out test fonts:", m["test_fonts"])
print("Most common mistakes:", m["top_confusions_heldout"][:6])
display(Image("docs/training_curves.png")); display(Image("docs/confusion_matrix.png"))

## 4 · Run the full pipeline on the sample images

In [ ]:
import cv2, glob, os, matplotlib.pyplot as plt
from anpr.pipeline import run, annotate
paths = sorted(glob.glob("samples/*"))
plt.figure(figsize=(16, 8))
for i, p in enumerate(paths):
    r = run(cv2.imread(p), mode="auto")
    plt.subplot(2, 3, i + 1)
    plt.imshow(cv2.cvtColor(annotate(r), cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"{r.decoded.pretty()}  ({r.confidence:.0%})", fontsize=13)
    print(f"{os.path.basename(p):36s} -> {r.decoded.pretty():16s} valid={r.decoded.valid}  state={r.decoded.state}")
plt.tight_layout(); plt.show()

## 5 · Try your own photo

In [ ]:
from google.colab import files
up = files.upload()
for name, data in up.items():
    open(name, "wb").write(data)
    r = run(cv2.imread(name), mode="auto")
    plt.figure(figsize=(10, 6)); plt.imshow(cv2.cvtColor(annotate(r), cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"{r.decoded.pretty()}  valid={r.decoded.valid}  ({r.confidence:.0%})"); plt.show()

## 6 · Tests, then download the new model
To use your re-trained model in the Streamlit app, upload these files to the `models/` and `docs/` folders of your GitHub repository (replacing the old ones).

In [ ]:
!python -m pytest -q tests
from google.colab import files
for f in ["models/char_cnn.onnx", "models/metrics.json", "docs/training_curves.png", "docs/confusion_matrix.png"]:
    files.download(f)